In [ ]:
import os
import numpy as np
import pandas as pd
import pandas_gbq
from google.cloud import bigquery

WORKSPACE_CDR = 'wb-affable-acorn-7941.R2024Q3R8'
print(WORKSPACE_CDR)

pd.set_option('max_colwidth', 800) # show all contents

In [ ]:
query=f"""
WITH doc_count AS (
  SELECT 
    count(condition_concept_id) as count_terms,
    extract(year from condition_era_start_date) as start_year,
    person_id
  FROM {WORKSPACE_CDR}.condition_era
  GROUP BY person_id, start_year
), 

bins AS (
  select
    -- CAST AS INT rounds to nearest, so these break in the middle
    10 * CAST((count_terms) / 10.0 AS INT64) as count_binned,
  from doc_count
)

select
  count_binned,
  count(*) as num_bins
from bins
GROUP BY count_binned
ORDER BY count_binned
LIMIT 30
"""
df = pandas_gbq.read_gbq(query, dialect='standard')
df.head(20)

In [ ]:
# no year binning

query=f"""
WITH doc_count AS (
  SELECT 
    count(condition_concept_id) as count_terms,
    person_id
  FROM {WORKSPACE_CDR}.condition_era
  GROUP BY person_id
), 

bins AS (
  select
    -- CAST AS INT rounds to nearest, so these break in the middle
    10 * CAST((count_terms) / 10.0 AS INT64) as count_binned,
  from doc_count
)

select
  count_binned,
  count(*) as num_bins
from bins
GROUP BY count_binned
ORDER BY count_binned
LIMIT 30
"""
df = pandas_gbq.read_gbq(query, dialect='standard')
df.head(20)

In [ ]:
# year-binned with era replication across span; histogram + cumulative tail.
# Replicating an era across every calendar year it covers gives "active in
# year Y" semantics: a chronic-HTN era from 2010-2018 contributes 9 rows.
# Patient-year docs aggregated from this match what CountVectorizer would
# see if each (patient, year) is its own document.

query = f"""
WITH
era_year_replicated AS (
  SELECT
    ce.person_id,
    ce.condition_concept_id,
    year_active
  FROM `{WORKSPACE_CDR}.condition_era` ce
  CROSS JOIN UNNEST(
    GENERATE_ARRAY(
      EXTRACT(YEAR FROM ce.condition_era_start_date),
      -- Clip future-dated end dates to the current year so the array stays
      -- bounded against any pathological future-dating in the source.
      LEAST(
        EXTRACT(YEAR FROM ce.condition_era_end_date),
        EXTRACT(YEAR FROM CURRENT_DATE())
      )
    )
  ) AS year_active
),
person_year_term_counts AS (
  SELECT
    person_id,
    year_active,
    COUNT(condition_concept_id) AS count_terms
  FROM era_year_replicated
  GROUP BY person_id, year_active
),
binned AS (
  SELECT
    -- CAST AS INT64 rounds to nearest: bin 0 holds 0..4, bin 10 holds 5..14, etc.
    10 * CAST(count_terms / 10.0 AS INT64) AS count_binned
  FROM person_year_term_counts
),
histogram AS (
  SELECT
    count_binned,
    COUNT(*) AS num_docs
  FROM binned
  GROUP BY count_binned
)
SELECT
  count_binned,
  num_docs,
  -- Descending cumulative: num_docs_at_or_above[C] = how many docs have
  -- at least C terms. Read as post-filter cohort size if you drop bins
  -- shorter than C.
  SUM(num_docs) OVER (ORDER BY count_binned DESC) AS num_docs_at_or_above,
  ROUND(
    SUM(num_docs) OVER (ORDER BY count_binned DESC) * 100.0
    / SUM(num_docs) OVER (),
    2
  ) AS pct_at_or_above
FROM histogram
ORDER BY count_binned
"""
df = pandas_gbq.read_gbq(query, dialect='standard')
df.head(40)
